# 1st LAB: Word Embeddings

In [1]:
import re
import string

import nltk
import numpy as np
import pandas as pd

# Loading the data

In [2]:
# US presidential inaugural speeches, from the NLTK corpora
nltk.download('inaugural') ###
from nltk.corpus import inaugural

[nltk_data] Downloading package inaugural to /root/nltk_data...
[nltk_data]   Package inaugural is already up-to-date!


In [3]:
# Extract the year, president's name and speech text from each file
data = pd.DataFrame({
    # Extract the year from the first 4 characters of each filename
    'year': [int(f[:4]) for f in inaugural.fileids()],
    # Extract the president's name from 6th char to last before ".txt"
    'president': [f[5:-4] for f in inaugural.fileids()],
    # Retrieve the full text of each inaugural speech
    'text': [inaugural.raw(f) for f in inaugural.fileids()],
}).sort_values('year', ignore_index=True)  # Sort speeches by year

data.head()

,year,president,text
0,1789,Washington,Fellow-Citizens of the Senate and of the House...
1,1793,Washington,"Fellow citizens, I am again called upon by the..."
2,1797,Adams,"When it was first perceived, in early times, t..."
3,1801,Jefferson,Friends and Fellow Citizens:\n\nCalled upon to...
4,1805,Jefferson,"Proceeding, fellow citizens, to that qualifica..."


In [4]:
# Define functions to remove punctuation and numbers from text
def remove_punctuation(text):
    return re.sub(f"[{re.escape(string.punctuation)}]", " ", text)

def remove_numbers(text):
    return re.sub(r"\d+", "", text)

In [5]:
# Convert text to lowercase
data["preprocessed"] = data["text"].str.lower()

# Remove numbers and punctuation
data["preprocessed"] = data["preprocessed"].apply(remove_numbers)
data["preprocessed"] = data["preprocessed"].apply(remove_punctuation)

# Split text into individual words (tokens)
data["tokens"] = data["preprocessed"].str.split()

data.head()

,year,president,text,preprocessed,tokens
0,1789,Washington,Fellow-Citizens of the Senate and of the House...,fellow citizens of the senate and of the house...,"[fellow, citizens, of, the, senate, and, of, t..."
1,1793,Washington,"Fellow citizens, I am again called upon by the...",fellow citizens i am again called upon by the...,"[fellow, citizens, i, am, again, called, upon,..."
2,1797,Adams,"When it was first perceived, in early times, t...",when it was first perceived in early times t...,"[when, it, was, first, perceived, in, early, t..."
3,1801,Jefferson,Friends and Fellow Citizens:\n\nCalled upon to...,friends and fellow citizens \n\ncalled upon to...,"[friends, and, fellow, citizens, called, upon,..."
4,1805,Jefferson,"Proceeding, fellow citizens, to that qualifica...",proceeding fellow citizens to that qualifica...,"[proceeding, fellow, citizens, to, that, quali..."


In [6]:
print(len(data))

60


# Introduction to gensim

Before the exercises, let's see **how to use** the gensim tools on a toy corpus of 5 sentences.

> **BEWARE:** the corpus is far too small, so the results (vectors, similar words) mean nothing. Look only at **which functions** we call and **in which form** we pass the data.

In [7]:
from gensim.models import Word2Vec

# Word2Vec wants a LIST OF SENTENCES, each sentence a LIST OF WORDS
toy = [
    ['i', 'like', 'green', 'eggs', 'and', 'ham'],
    ['green', 'eggs', 'for', 'breakfast'],
    ['i', 'like', 'bacon', 'and', 'toast'],
    ['we', 'eat', 'green', 'eggs', 'with', 'toast'],
    ['the', 'king', 'eats', 'ham', 'and', 'bacon'],
]

# sg=1: skip-gram, sg=0: CBOW
# min_count=1: keep also the words seen only once (the default, 5, would remove all of them here)
toy_model = Word2Vec(sentences=toy, vector_size=10, window=2, min_count=1, sg=1, seed=42)

print('Vocabulary:', list(toy_model.wv.key_to_index))
print('Vector of eggs:', toy_model.wv['eggs'].round(2))
print('Most similar to eggs:', toy_model.wv.most_similar('eggs', topn=3))

# word analogies: positive words are added, negative words are subtracted (eggs + bacon - ham)
print('eggs + bacon - ham:', toy_model.wv.most_similar(positive=['eggs', 'bacon'], negative=['ham'], topn=3))

# a word that is not in the corpus has no vector
print("Is 'pizza' in the vocabulary?", 'pizza' in toy_model.wv)

Vocabulary: ['and', 'eggs', 'green', 'toast', 'bacon', 'ham', 'like', 'i', 'eats', 'king', 'the', 'with', 'eat', 'we', 'breakfast', 'for']
Vector of eggs: [ 0.01  0.1   0.05  0.05  0.04  0.06  0.   -0.07  0.07 -0.01]
Most similar to eggs: [('king', 0.3018323481082916), ('with', 0.2954557240009308), ('eats', 0.2678139805793762)]
eggs + bacon - ham: [('and', 0.6240828037261963), ('eat', 0.36743733286857605), ('with', 0.24361400306224823)]
Is 'pizza' in the vocabulary? False


### Bigrams with `Phrases`

`Phrases` finds the pairs of words that often appear together and joins them with `_`.

In [8]:
from gensim.models import Phrases

# min_count and threshold are very low only because the toy corpus is tiny: on real data keep the defaults
toy_phrases = Phrases(toy, min_count=1, threshold=1)

# apply it to the sentences: the result is a new corpus, in the same form (list of lists of words)
toy_bigrams = list(toy_phrases[toy])
print(toy_bigrams)

[['i_like', 'green_eggs', 'and', 'ham'], ['green_eggs', 'for', 'breakfast'], ['i_like', 'bacon', 'and', 'toast'], ['we', 'eat', 'green_eggs', 'with', 'toast'], ['the', 'king', 'eats', 'ham', 'and', 'bacon']]


### Document vectors with `Doc2Vec`

`Doc2Vec` wants each document as a `TaggedDocument`: the list of its words + a tag (here, its number).

In [9]:
from gensim.models.doc2vec import Doc2Vec, TaggedDocument

toy_docs = [TaggedDocument(words, [i]) for i, words in enumerate(toy)]
print(toy_docs[0])

toy_doc_model = Doc2Vec(toy_docs, vector_size=10, window=2, min_count=1, seed=42)

# the vector learned for a document of the corpus, by its tag
print('Vector of document 0:', toy_doc_model.dv[0].round(2))

# the vector of a NEW text (a list of words, not a string)
print('Vector of a new text:', toy_doc_model.infer_vector(['green', 'eggs', 'and', 'bacon']).round(2))

TaggedDocument<['i', 'like', 'green', 'eggs', 'and', 'ham'], [0]>
Vector of document 0: [ 0.08  0.09  0.08 -0.08 -0.08 -0.05 -0.04  0.05 -0.07 -0.08]
Vector of a new text: [-0.03 -0.02 -0.05 -0.   -0.    0.04  0.    0.05 -0.03 -0.02]


# Exercise 3.1: Static Word Embeddings

---




Steps:

Part 1 **Training Word2vec on a dataset**:
- Train a Word2Vec model on the presidents' speeches. You must pass to Word2Vec a list of lists of tokens (a list of tokenized sentences). Use CBOW and a window of 5. Documentation: https://radimrehurek.com/gensim/models/word2vec.html#gensim.models.word2vec.Word2Vec
- Find the words most similar to, e.g. "state", "states", "president", or others.
- Compare the effectiveness of CBOW and Skipgram. Which one works better?


Part 2 **Using pre-trained static models**:
- Load a pre-trained gensim model (available models: https://github.com/RaRe-Technologies/gensim-data#Models) (Choose a small one to avoid waiting for the download, e.g. `glove-wiki-gigaword-100`).
- Evaluate the following relations:
  - king + woman - man = ?
  - France - Paris + London = ?
  - Find out some for yourselves

**Question: Find the words most similar to "italy" using the 2 models (trained and pretrained). What do you observe? Why? Any solution?**



In [7]:
# Your code here

from gensim import downloader
from gensim.models import Word2Vec

In [8]:
nltk.download('punkt') ###< run once
nltk.download('punkt_tab') ###< run once, needed by recent NLTK versions

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [9]:
data['sent_tokens'] = data.text.str.lower()

data.sent_tokens = data.sent_tokens.str.replace(r'\d+', ' ', regex=True)
data.sent_tokens = data.sent_tokens.apply(lambda input_text : [remove_punctuation(t).split() for t in nltk.sent_tokenize(input_text)])

In [10]:
print('Number of sentences:', len(data.sent_tokens.sum()))

Number of sentences: 5360


In [11]:
print('Number of speeches: ', len(data.sent_tokens))

Number of speeches:  60


In [12]:
corpus = data.sent_tokens.sum() # a list of sentences (each one a list of words), instead of a list of speeches
# now use the slides code for training!

# sg=0 for CBOW, sg=1 for skip-gram
cbow_model = Word2Vec(sentences=corpus, vector_size=100, window=5, sg=0, seed=42)
model = Word2Vec(sentences=corpus, vector_size=100, window=5, sg=1, seed=42)

# set similar words to:
# states, war, state
print('CBOW:     ', [word for word, score in cbow_model.wv.most_similar('states', topn=8)])
print('Skip-gram:', [word for word, score in model.wv.most_similar('states', topn=8)])

# try also italy and see what happens!

CBOW:      ['of', 'the', 'united', 'constitution', 'people', 'world', 'and', 'government']
Skip-gram: ['constitution', 'powers', 'protect', 'defend', 'authority', 'territories', 'federal', 'reserved']


> **Note: your results may differ.** `seed=42` fixes the starting point, but gensim trains with several threads in parallel, and the order in which the threads work changes at every run. So the similar words (and their scores) can change a little from one run to another, and from the ones of your classmates. The same holds for `Doc2Vec` in Exercise 3.4.
>
> Look at the *kind* of words you get, not at the exact list.

In [13]:
# 'italy' never appears in the speeches, so it has no vector
print("Is 'italy' in the vocabulary?", 'italy' in model.wv)

Is 'italy' in the vocabulary? False


In [14]:
downloader.info()['models'].keys()

dict_keys(['fasttext-wiki-news-subwords-300', 'conceptnet-numberbatch-17-06-300', 'word2vec-ruscorpora-300', 'word2vec-google-news-300', 'glove-wiki-gigaword-50', 'glove-wiki-gigaword-100', 'glove-wiki-gigaword-200', 'glove-wiki-gigaword-300', 'glove-twitter-25', 'glove-twitter-50', 'glove-twitter-100', 'glove-twitter-200', '__testing_word2vec-matrix-synopsis'])

In [15]:
wiki_model = downloader.load('glove-wiki-gigaword-100')

In [16]:
president_vector = wiki_model['president']
print('Vector size:', president_vector.shape)
print('First 5 values:', president_vector[:5])

Vector size: (100,)
First 5 values: [-0.064549 -0.13812   0.50017   0.41434   0.45832 ]


In [17]:
wiki_model.most_similar(positive=['king', 'woman'], negative=['man'], topn=5)

[('queen', 0.7698540687561035),
 ('monarch', 0.6843381524085999),
 ('throne', 0.6755736470222473),
 ('daughter', 0.6594556570053101),
 ('princess', 0.6520534157752991)]

In [18]:
wiki_model.most_similar(positive=['france', 'london'], negative=['paris'], topn=5)

[('britain', 0.8921419978141785),
 ('england', 0.8002905249595642),
 ('australia', 0.7940487265586853),
 ('scotland', 0.7838966846466064),
 ('british', 0.7608305215835571)]

In [19]:
# 'italy' is not in our speeches, but it is in the pre-trained model (trained on Wikipedia)
wiki_model.most_similar('italy', topn=5)

[('spain', 0.7746186256408691),
 ('italian', 0.7569283246994019),
 ('portugal', 0.7421525716781616),
 ('germany', 0.740085244178772),
 ('greece', 0.7235245108604431)]

### Answers

**CBOW or skip-gram?** Look at the words most similar to `states`:
- CBOW returns many stop words (`the`, `of`, `in`): they appear next to `states` very often, and CBOW learns mostly from frequent words.
- Skip-gram returns words related to the meaning (`constitution`, `powers`, `federal`, `territories`).

On a small corpus (about 5,000 sentences) **skip-gram works better**, because it learns a good vector also for the rare words.

**What about `italy`?**
- Trained model: `italy` never appears in the speeches, so it has **no vector** (`'italy' in model.wv` is `False`, and `most_similar('italy')` gives a `KeyError`).
- Pre-trained GloVe: it was trained on Wikipedia, so it knows `italy`, and the most similar words are other countries (`spain`, `portugal`, `germany`, `greece`) and `italian`.

**Solutions:** use a pre-trained model, trained on a much larger corpus; or a model that builds the vector from pieces of the word (e.g. *fastText*, `fasttext-wiki-news-subwords-300`), which gives a vector also to words it has never seen.

# Exercise 3.2: Embeddings with multiword ngrams

---



**Training Word2vec on a dataset**:
- Train a bigram detector using gensim.
- Train a Word2Vec model on the presidents' speeches. You must pass to Word2Vec a list of lists of bigram tokens (a list of tokenized sentences).
- Find the words most similar to, e.g. "united_states", "american_people" or others.
- Try both word2vec architectures.

**Have you seen the sparsity problem?**

In [20]:
from gensim.models import Phrases

# train the bigram detector on the sentences (the same corpus as in Exercise 3.1)
bigram_generator = Phrases(corpus)

In [21]:
bigram_tokens = list(bigram_generator[corpus])

# how many times does each bigram appear? (fewer occurrences = less to learn from)
bigram_counts = pd.Series([w for sentence in bigram_tokens for w in sentence]).value_counts()
print(bigram_counts[['united_states', 'american_people']])

united_states      166
american_people     41
Name: count, dtype: int64


In [22]:
# use bigram_tokens as your new corpus (same code as before!)
model = Word2Vec(sentences=bigram_tokens, vector_size=100, window=5, sg=1, seed=42)

In [23]:
print(model.wv.most_similar('united_states', topn=10))

[('constitution', 0.9698435664176941), ('executive', 0.9406654238700867), ('body', 0.9403587579727173), ('legislative', 0.9371962547302246), ('policy', 0.9368451833724976), ('whole', 0.9350869655609131), ('general_government', 0.9348016381263733), ('first', 0.9326164722442627), ('protect', 0.9305777549743652), ('welfare', 0.9273220300674438)]


In [24]:
print(model.wv.most_similar('american_people', topn=10))

[('evils', 0.9911245107650757), ('amendment', 0.9910457730293274), ('hands', 0.9902584552764893), ('expressed', 0.9897894263267517), ('officer', 0.9889227151870728), ('obligation', 0.988735020160675), ('responsibilities', 0.9885676503181458), ('obedience', 0.9885120391845703), ('depend', 0.9883927702903748), ('execute', 0.9883227348327637)]


In [25]:
# The same with CBOW (sg=0)
cbow_bigram_model = Word2Vec(sentences=bigram_tokens, vector_size=100, window=5, sg=0, seed=42)
print('CBOW:     ', [w for w, s in cbow_bigram_model.wv.most_similar('american_people', topn=8)])
print('Skip-gram:', [w for w, s in model.wv.most_similar('american_people', topn=8)])

# How similar are the neighbours? Look at the scores
print('Skip-gram scores:', [round(s, 3) for w, s in model.wv.most_similar('american_people', topn=8)])

CBOW:      ['same', 'office', 'new', 'should_be', 'through', 'having', 'too', 'high']
Skip-gram: ['evils', 'amendment', 'hands', 'expressed', 'officer', 'obligation', 'responsibilities', 'obedience']
Skip-gram scores: [0.991, 0.991, 0.99, 0.99, 0.989, 0.989, 0.989, 0.989]


### Answer: the sparsity problem

Joining two words into one token makes each token **rarer**: `american_people` appears only about 40 times in all the speeches (see the counts above), while `american` and `people` alone appear many more times.

With so few examples the model cannot learn a good vector:
- the most similar words to `american_people` have nothing to do with its meaning;
- their scores are all very high and almost identical (≈ 0.99): the model cannot tell them apart.

`united_states` (about 160 times) works a bit better. The more often a token appears, the better its vector. To fix it we need a **bigger corpus**, or a pre-trained model.

# Exercise 3.3

You can represent a document as the weighted sum of the word embeddings of the terms it contains, where the term weights are given by, e.g. Tf-Idf.

- Compute the document embeddings for the US presidential speeches corpus.
- Write a function that, given a document (speech) id, finds the top n most similar documents (adapt the previous one for most similar terms).


Hints (but there may be other / better solutions):

- To obtain the TfIdf values from the 'tfidf' matrix returned by fit_transform, use 'tfidf[doc_id, term_index]'. The list of terms (features) is returned by 'vectorizer.get_feature_names_out()'
- To obtain the Word2Vec embedding for a word, use 'model.wv[word]', where 'model = Word2Vec(...)'
- To obtain the document embedding, iterate over all of its words (that are present both as features for the TfIdf vectorizer and in the Word2Vec model vocabulary), and sum over all the products: (model.wv[word] * tfidf[doc_id, word_index])

In [26]:
# Your code here

In [27]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer()
docs_tfidf = vectorizer.fit_transform(data.preprocessed)

model = Word2Vec(sentences=corpus, window=5, sg=1, seed=42) # we use skipgram

In [28]:
tf_idf_names = vectorizer.get_feature_names_out()

In [29]:
skip_gram_words = model.wv.key_to_index

In [30]:
tfidf_feature_to_index = {word: i for i, word in enumerate(tf_idf_names)}

def tfidf_weight(doc_id, term):
    return docs_tfidf[doc_id, tfidf_feature_to_index[term]]

In [31]:
skip_gram_set = set(skip_gram_words)
tfidf_set = set(tf_idf_names)

common_terms = skip_gram_set.intersection(tfidf_set)

In [32]:
def get_document_embedding(doc_id, common_terms):
    document = data.tokens[doc_id]
    embedding = np.zeros(model.vector_size)
    for word in set(document):   # each word once: the TF-IDF weight already counts its repetitions
        if word in common_terms:
            embedding = embedding + model.wv[word] * tfidf_weight(doc_id, word)

    return embedding

In [33]:
doc_embeddings = np.zeros((data.shape[0], model.vector_size))

for i in range(data.shape[0]):
    doc_embeddings[i] = get_document_embedding(i, common_terms)

In [34]:
doc_embeddings.shape

(60, 100)

In [35]:
doc_cos = cosine_similarity(doc_embeddings, doc_embeddings)

def top_n_most_similar_document(doc_id, doc_sim, top_n=5):
    current_doc_sim = doc_sim[doc_id]
    best = (-current_doc_sim).argsort()[1:(top_n+1)]   # position 0 is the document itself
    return data[['year', 'president']].loc[best].assign(similarity=current_doc_sim[best].round(3))

In [36]:
doc_cos.shape

(60, 60)

In [37]:
data.loc[2, ['year', 'president']]

year          1797
president    Adams
Name: 2, dtype: object

In [38]:
top_n_most_similar_document(2, doc_cos, 5)

,year,president,similarity
30,1909,Taft,1.000
25,1889,Harrison,1.000
27,1897,McKinley,1.000
35,1929,Hoover,0.999
23,1881,Garfield,0.999


In [39]:
# Are the speeches really different? Look at the similarities between different speeches
others = doc_cos[~np.eye(len(doc_cos), dtype=bool)]   # all the values except the diagonal
print('Min similarity: ', others.min().round(3))
print('Mean similarity:', others.mean().round(3))

Min similarity:  0.963
Mean similarity: 0.995


All the speeches look almost identical to each other: they all use the same frequent words, so the sums of their vectors point in almost the same direction.

A simple fix: subtract the **mean vector** of all the speeches. What is left is what makes each speech different from the others.

In [40]:
doc_embeddings_centered = doc_embeddings - doc_embeddings.mean(axis=0)
doc_cos_centered = cosine_similarity(doc_embeddings_centered, doc_embeddings_centered)

others = doc_cos_centered[~np.eye(len(doc_cos_centered), dtype=bool)]
print('Min similarity: ', others.min().round(3))
print('Mean similarity:', others.mean().round(3))

# Compare the most similar speeches to Biden 2021, before and after centering
biden_id = data.index[data.year == 2021][0]
display(top_n_most_similar_document(biden_id, doc_cos))
display(top_n_most_similar_document(biden_id, doc_cos_centered))

Min similarity:  -0.989
Mean similarity: -0.012


,year,president,similarity
50,1989,Bush,0.999
39,1945,Roosevelt,0.999
51,1993,Clinton,0.998
48,1981,Reagan,0.998
59,2025,Trump,0.998


,year,president,similarity
50,1989,Bush,0.980
48,1981,Reagan,0.950
59,2025,Trump,0.950
55,2009,Obama,0.934
43,1961,Kennedy,0.931


# Exercise 3.4

Use Doc2Vec (https://radimrehurek.com/gensim/models/doc2vec.html) to compute the embeddings for the documents in the presidential speeches dataset. Write a function that finds the most similar documents for a given document(adapt the previous function).

In [41]:
# Your code here

from gensim.models.doc2vec import Doc2Vec, TaggedDocument

In [42]:
documents = [TaggedDocument(doc, [i]) for i, doc in enumerate(data.tokens)]

In [43]:
doc_model = Doc2Vec(documents, window=5, seed=42)

In [44]:
# generic method to get an emebedding not based only on the document id
# d2v_doc_embeddings = np.array([doc_model.infer_vector(data.tokens[i]) for i in range(data.shape[0])])


# use the static embedding generated by doc2vec model
d2v_doc_embeddings = np.array([doc_model.dv[i] for i in range(data.shape[0])])

In [45]:
d2v_doc_embeddings.shape

(60, 100)

In [46]:
d2v_doc_cos = cosine_similarity(d2v_doc_embeddings, d2v_doc_embeddings)

In [47]:
top_n_most_similar_document(2, d2v_doc_cos)

,year,president,similarity
7,1817,Monroe,0.971
22,1877,Hayes,0.962
35,1929,Hoover,0.960
14,1845,Polk,0.959
5,1809,Madison,0.958


In [48]:
ex_query = 'fellow citizens of the united states of america'
# infer the query (or a doc) vector using only the words (so good for new documents or queries)
# infer_vector wants a list of words, not a string
q = doc_model.infer_vector(ex_query.split())

# print(cosine_similarity(d2v_doc_embeddings, q.reshape(1, -1)))
cosine_similarity(d2v_doc_embeddings, q.reshape(1, -1)).shape

(60, 1)

In [49]:
def get_top_n_from_query(query, doc_embeddings, top_n=5):
    q = doc_model.infer_vector(query.split())
    similarities = cosine_similarity(doc_embeddings, q.reshape(1, -1)).flatten()
    # one row per speech, sorted from the most similar
    ranking = data[['year', 'president']].assign(similarity=similarities.round(3))
    return ranking.sort_values('similarity', ascending=False).head(top_n)

# infer_vector is random: we save the result to use it again below
top_docs = get_top_n_from_query(ex_query, d2v_doc_embeddings)
top_docs

,year,president,similarity
58,2021,Biden,0.887
45,1969,Nixon,0.858
59,2025,Trump,0.858
48,1981,Reagan,0.845
50,1989,Bush,0.842


In [50]:
ex_query

'fellow citizens of the united states of america'

In [51]:
# beginning of the most similar speech
top_id = top_docs.index[0]
print(data.loc[top_id, 'year'], data.loc[top_id, 'president'])
print(data.loc[top_id, 'text'][:200], '...')

2021 Biden
Chief Justice Roberts, Vice President Harris, Speaker Pelosi, Leader Schumer, Leader McConnell, Vice President Pence, and my distinguished guests, and my fellow Americans: This is America's day. This  ...
